# Email Feature Engineering

This notebook processes the complete email dataset using chunk processing to avoid memory issues.

# 03_Email_Feature_Engineering.ipynb

## Objective

In this notebook, we will engineer employee-level features from the **email.csv** dataset.

We will:

- Load the email dataset
- Explore its structure
- Clean and preprocess the data
- Engineer behavioral features
- Aggregate features at the employee level
- Save the final employee email profile

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
DATA_PATH = Path("../DATA/raw/CERT_R6.2/data")
FEATURE_PATH = Path("../DATA/features")

FEATURE_PATH.mkdir(parents=True, exist_ok=True)

In [3]:
#verify dataset path
print("Current Working Directory:")
print(Path.cwd())

print("\nDataset Path:")
print(DATA_PATH)

print("\nFolder Exists:")
print(DATA_PATH.exists())

print("\nEmail File Exists:")
print((DATA_PATH / "email.csv").exists())

Current Working Directory:
d:\OPENCODE\insider-threat-detection\NOTEBOOKS

Dataset Path:
..\DATA\raw\CERT_R6.2\data

Folder Exists:
True

Email File Exists:
True


In [4]:
chunks = pd.read_csv(
    DATA_PATH / "email.csv",
    chunksize=100000
)

print("Chunk Reader Created Successfully")

Chunk Reader Created Successfully


In [5]:
email_profiles = []

for chunk_number, chunk in enumerate(chunks, start=1):

    print(f"Processing Chunk {chunk_number}")

    chunk["date"] = pd.to_datetime(
        chunk["date"],
        format="%m/%d/%Y %H:%M:%S"
    )

    chunk["hour"] = chunk["date"].dt.hour

    chunk["weekday"] = chunk["date"].dt.weekday

    chunk["after_hours_email"] = (
        (chunk["hour"] < 8) |
        (chunk["hour"] >= 18)
    ).astype(int)

    chunk["weekend_email"] = (
        chunk["weekday"] >= 5
    ).astype(int)

    chunk["has_attachment"] = (
        chunk["attachments"]
        .fillna("")
        .str.strip()
        .ne("")
    ).astype(int)

    chunk["email_size_kb"] = chunk["size"] / 1024

    chunk["recipient_count"] = (
        chunk["to"]
        .fillna("")
        .apply(
            lambda x: len(
                [r for r in str(x).split(";") if r.strip() != ""]
            )
        )
    )

    chunk["is_sent"] = (
        chunk["activity"] == "Send"
    ).astype(int)

    chunk_profile = chunk.groupby("user").agg(

        total_email_events=("activity", "count"),
        total_email_sent=("is_sent", "sum"),
        after_hours_email_count=("after_hours_email", "sum"),
        weekend_email_count=("weekend_email", "sum"),
        attachment_email_count=("has_attachment", "sum"),
        total_email_size_kb=("email_size_kb", "sum"),
        total_recipients=("recipient_count", "sum"),
        email_count=("activity", "count")

    ).reset_index()

    email_profiles.append(chunk_profile)

Processing Chunk 1
Processing Chunk 2
Processing Chunk 3
Processing Chunk 4
Processing Chunk 5
Processing Chunk 6
Processing Chunk 7
Processing Chunk 8
Processing Chunk 9
Processing Chunk 10
Processing Chunk 11
Processing Chunk 12
Processing Chunk 13
Processing Chunk 14
Processing Chunk 15
Processing Chunk 16
Processing Chunk 17
Processing Chunk 18
Processing Chunk 19
Processing Chunk 20
Processing Chunk 21
Processing Chunk 22
Processing Chunk 23
Processing Chunk 24
Processing Chunk 25
Processing Chunk 26
Processing Chunk 27
Processing Chunk 28
Processing Chunk 29
Processing Chunk 30
Processing Chunk 31
Processing Chunk 32
Processing Chunk 33
Processing Chunk 34
Processing Chunk 35
Processing Chunk 36
Processing Chunk 37
Processing Chunk 38
Processing Chunk 39
Processing Chunk 40
Processing Chunk 41
Processing Chunk 42
Processing Chunk 43
Processing Chunk 44
Processing Chunk 45
Processing Chunk 46
Processing Chunk 47
Processing Chunk 48
Processing Chunk 49
Processing Chunk 50
Processin

In [6]:
# Combine All Processed Chunks
email_profile = pd.concat(
    email_profiles,
    ignore_index=True
)

print("Total Rows:", len(email_profile))
email_profile.head()

Total Rows: 421654


,user,total_email_events,total_email_sent,after_hours_email_count,weekend_email_count,attachment_email_count,total_email_size_kb,total_recipients,email_count
0,AAB0162,32,15,8,0,4,8456.549805,57,32
1,AAB0398,38,10,3,0,6,13402.002930,59,38
2,AAC0610,9,4,0,0,3,3099.632812,15,9
3,AAC0668,27,8,0,0,6,8222.534180,41,27
4,AAC3270,3,1,0,0,0,86.383789,3,3


In [7]:
# Merge Duplicate Users Across Chunks
email_profile = (
    email_profile
    .groupby("user", as_index=False)
    .sum()
)

print("Unique Users:", len(email_profile))
email_profile.head()

Unique Users: 4000


,user,total_email_events,total_email_sent,after_hours_email_count,weekend_email_count,attachment_email_count,total_email_size_kb,total_recipients,email_count
0,AAB0162,3146,1122,440,0,590,1.101054e+06,6027,3146
1,AAB0398,3811,1348,698,0,1223,2.239566e+06,7201,3811
2,AAC0610,1052,388,16,0,375,5.583907e+05,2103,1052
3,AAC0668,3115,1093,23,0,689,1.177002e+06,6168,3115
4,AAC3270,344,120,0,0,170,2.913571e+05,682,344


In [8]:
# Calculate Average Features
email_profile["average_email_size_kb"] = (
    email_profile["total_email_size_kb"] /
    email_profile["email_count"]
)

email_profile["average_recipient_count"] = (
    email_profile["total_recipients"] /
    email_profile["email_count"]
)

In [9]:
# Remove Temporary Columns
email_profile.drop(
    columns=[
        "total_email_size_kb",
        "total_recipients",
        "email_count"
    ],
    inplace=True
)

email_profile.head()

,user,total_email_events,total_email_sent,after_hours_email_count,weekend_email_count,attachment_email_count,average_email_size_kb,average_recipient_count
0,AAB0162,3146,1122,440,0,590,349.985317,1.915766
1,AAB0398,3811,1348,698,0,1223,587.658355,1.889530
2,AAC0610,1052,388,16,0,375,530.789627,1.999049
3,AAC0668,3115,1093,23,0,689,377.849683,1.980096
4,AAC3270,344,120,0,0,170,846.968435,1.982558


In [10]:
# Save Email Features
output_file = FEATURE_PATH / "email_features.csv"

email_profile.to_csv(output_file, index=False)

print("Email features saved successfully.")
print(output_file)

Email features saved successfully.
..\DATA\features\email_features.csv


In [11]:
# Verify Saved Dataset
saved_df = pd.read_csv(output_file)

print(saved_df.shape)
saved_df.head()

(4000, 8)


,user,total_email_events,total_email_sent,after_hours_email_count,weekend_email_count,attachment_email_count,average_email_size_kb,average_recipient_count
0,AAB0162,3146,1122,440,0,590,349.985317,1.915766
1,AAB0398,3811,1348,698,0,1223,587.658355,1.889530
2,AAC0610,1052,388,16,0,375,530.789627,1.999049
3,AAC0668,3115,1093,23,0,689,377.849683,1.980096
4,AAC3270,344,120,0,0,170,846.968435,1.982558


In [12]:
email_profile.shape

(4000, 8)

In [13]:
email_profile.head()

,user,total_email_events,total_email_sent,after_hours_email_count,weekend_email_count,attachment_email_count,average_email_size_kb,average_recipient_count
0,AAB0162,3146,1122,440,0,590,349.985317,1.915766
1,AAB0398,3811,1348,698,0,1223,587.658355,1.889530
2,AAC0610,1052,388,16,0,375,530.789627,1.999049
3,AAC0668,3115,1093,23,0,689,377.849683,1.980096
4,AAC3270,344,120,0,0,170,846.968435,1.982558


In [14]:
saved_df = pd.read_csv("../DATA/features/email_features.csv")

print(saved_df.shape)
print(saved_df.head())

(4000, 8)
      user  total_email_events  total_email_sent  after_hours_email_count  \
0  AAB0162                3146              1122                      440   
1  AAB0398                3811              1348                      698   
2  AAC0610                1052               388                       16   
3  AAC0668                3115              1093                       23   
4  AAC3270                 344               120                        0   

   weekend_email_count  attachment_email_count  average_email_size_kb  \
0                    0                     590             349.985317   
1                    0                    1223             587.658355   
2                    0                     375             530.789627   
3                    0                     689             377.849683   
4                    0                     170             846.968435   

   average_recipient_count  
0                 1.915766  
1                 1.889530  
2